### Calcular factores de reduccion por área para las cuencas

In [2]:
import os
import glob
import numpy as np
import pandas as pd


# ============================================================
# 1. RUTAS
# ============================================================

ruta_thiessen = r"C:\Users\DAVID01\Documents\GitHub\Geospatial-analysis\data\Thiessen_FRA.xlsx"
carpeta_series = r"C:\Users\DAVID01\Documents\GitHub\Geospatial-analysis\data\series limpias"
ruta_salida = r"salidas\FRA_cuencas.csv"


# ============================================================
# 2. CAMPOS
# ============================================================

# Campos en la tabla Thiessen_FRA.xlsx
campo_cuenca = "HYBAS_ID"
campo_estacion = "Codigo"
campo_wi = "Wi"

# Campos dentro de cada CSV diario
campo_fecha = "Fecha"
campo_lluvia = "Valor"


# ============================================================
# 3. PARÁMETROS
# ============================================================

min_dias_validos_anio = 256
cobertura_minima = 0.80


# ============================================================
# 4. LEER TABLA DE THIESSEN
# ============================================================

pesos = pd.read_excel(ruta_thiessen, sheet_name="interseccin")

pesos = pesos[[campo_cuenca, campo_estacion, campo_wi]].copy()

pesos[campo_cuenca] = pesos[campo_cuenca].astype(str)
pesos[campo_estacion] = pesos[campo_estacion].astype(str)
pesos[campo_wi] = pd.to_numeric(pesos[campo_wi], errors="coerce")

pesos = pesos.dropna(subset=[campo_cuenca, campo_estacion, campo_wi])

# Quitar estaciones con peso cero
pesos = pesos[pesos[campo_wi] > 0].copy()

# Si una estación aparece más de una vez dentro de una misma cuenca,
# se suman sus pesos.
pesos = (
    pesos
    .groupby([campo_cuenca, campo_estacion], as_index=False)[campo_wi]
    .sum()
)

# ============================================================
# 5. LEER SERIES DIARIAS
# ============================================================

def codigo_desde_archivo(ruta):
    """
    Extrae el código desde nombres tipo:
    27010830_San Antonio.csv
    """
    nombre = os.path.basename(ruta)
    nombre_sin_extension = os.path.splitext(nombre)[0]
    codigo = nombre_sin_extension.split("_", 1)[0]
    return str(codigo)


def leer_estacion(ruta):
    """
    Lee un CSV de una estación y devuelve una Serie:
    índice = fecha
    valor = precipitación diaria
    """
    codigo = codigo_desde_archivo(ruta)

    df = pd.read_csv(ruta)

    df[campo_fecha] = pd.to_datetime(df[campo_fecha], errors="coerce")
    df[campo_lluvia] = pd.to_numeric(df[campo_lluvia], errors="coerce")

    df = df.dropna(subset=[campo_fecha])
    df = df[[campo_fecha, campo_lluvia]].copy()
    df = df.set_index(campo_fecha).sort_index()

    # Si hay fechas repetidas, se promedia
    serie = df[campo_lluvia].groupby(df.index).mean()

    serie.name = codigo

    return serie


archivos = glob.glob(os.path.join(carpeta_series, "*.csv"))

series = []

for archivo in archivos:
    try:
        serie = leer_estacion(archivo)
        series.append(serie)
    except Exception as e:
        print("No se pudo leer:", archivo)
        print("Error:", e)

prec = pd.concat(series, axis=1)
prec = prec.sort_index()
prec.columns = prec.columns.astype(str)

print("Estaciones leídas:", prec.shape[1])
print("Periodo general:", prec.index.min(), "a", prec.index.max())


# ============================================================
# 6. FUNCIÓN PARA MÁXIMOS ANUALES
# ============================================================

def maximos_anuales(serie):
    """
    Extrae máximos anuales.
    Solo acepta años con al menos min_dias_validos_anio datos.
    """
    df = pd.DataFrame({"P": serie})
    df["anio"] = df.index.year

    resumen = df.groupby("anio")["P"].agg(["max", "count"])

    resumen = resumen[resumen["count"] >= min_dias_validos_anio]

    return resumen["max"].dropna()


# ============================================================
# 7. CALCULAR FRA PARA UNA CUENCA
# ============================================================

def calcular_fra_cuenca(id_cuenca):
    pesos_c = pesos[pesos[campo_cuenca] == str(id_cuenca)].copy()

    # Quedarse solo con estaciones que sí tienen archivo leído
    pesos_c = pesos_c[pesos_c[campo_estacion].isin(prec.columns)].copy()

    if pesos_c.empty:
        return {
            campo_cuenca: id_cuenca,
            "n_estaciones": 0,
            "R3_areal": np.nan,
            "R4_puntual": np.nan,
            "FRA": np.nan
        }

    estaciones = pesos_c[campo_estacion].tolist()

    w = pesos_c.set_index(campo_estacion)[campo_wi]
    w = w / w.sum()

    datos = prec[estaciones]

    # --------------------------------------------------------
    # R3: lluvia areal diaria y máximos anuales areales
    # --------------------------------------------------------

    cobertura = datos.notna().mul(w, axis=1).sum(axis=1)

    p_areal = datos.mul(w, axis=1).sum(axis=1, min_count=1)
    p_areal = p_areal / cobertura

    p_areal[cobertura < cobertura_minima] = np.nan

    max_areal = maximos_anuales(p_areal)

    R3 = max_areal.mean()

    # --------------------------------------------------------
    # R4: promedio ponderado de máximos anuales puntuales
    # --------------------------------------------------------

    maximos_medios_puntuales = {}

    for cod in estaciones:
        max_puntual = maximos_anuales(prec[cod])
        maximos_medios_puntuales[cod] = max_puntual.mean()

    maximos_medios_puntuales = pd.Series(maximos_medios_puntuales)

    R4 = (maximos_medios_puntuales * w).sum()

    # --------------------------------------------------------
    # FRA
    # --------------------------------------------------------

    FRA = R3 / R4 if R4 > 0 else np.nan

    return {
        campo_cuenca: id_cuenca,
        "n_estaciones": len(estaciones),
        "n_anios_areal": len(max_areal),
        "R3_areal": R3,
        "R4_puntual": R4,
        "FRA": FRA
    }


# ============================================================
# 8. CALCULAR FRA PARA TODAS LAS CUENCAS
# ============================================================

resultados = []

for id_cuenca in pesos[campo_cuenca].unique():
    resultados.append(calcular_fra_cuenca(id_cuenca))

fra = pd.DataFrame(resultados)

print(fra.head())
print(fra["FRA"].describe())


# ============================================================
# 9. EXPORTAR
# ============================================================

os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

fra.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo exportado:", ruta_salida)


C:\Users\DAVID01\AppData\Local\Temp\ipykernel_6100\1819802419.py:115: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  prec = pd.concat(series, axis=1)


Estaciones leídas: 223
Periodo general: 1931-07-01 00:00:00 a 2025-08-14 00:00:00
     HYBAS_ID  n_estaciones  n_anios_areal    R3_areal  R4_puntual       FRA
0  6080078120             5             35  113.938304  143.041639  0.796539
1  6080078150             7             28   79.371351  123.808470  0.641082
2  6080078690             2             39   77.709715   86.673751  0.896577
3  6080080190             6             33   85.465297  129.713203  0.658879
4  6080080350             4             42  102.332441  133.086661  0.768916
count    73.000000
mean      0.707724
std       0.139710
min       0.512460
25%       0.587501
50%       0.685320
75%       0.813743
max       1.000000
Name: FRA, dtype: float64
Archivo exportado: salidas\FRA_cuencas.csv
